In [1]:
# ====== ЭТАП 4: ОТБОР ПРИЗНАКОВ ======
import numpy as np
import pandas as pd
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, log_loss

# ====== ЗАГРУЗКА ДАННЫХ (уже есть) ======
# Используй X_train_feat, X_test_feat, y_train, y_test из предыдущего шага

# ====== МОДЕЛЬ ДЛЯ ОТБОРА ======
model_sfs = CatBoostClassifier(
    loss_function="Logloss",
    task_type="CPU",
    random_seed=0,
    iterations=200,
    verbose=False
)

# ====== SFS (Sequential Forward Selection) ======
sfs = SFS(
    model_sfs,
    k_features=15,
    forward=True,
    floating=False,
    scoring='roc_auc',
    cv=2,
    n_jobs=-1
)

print("⏳ Запуск SFS... (может занять 5–10 минут)")
sfs.fit(X_train_feat, y_train)

print("✅ Отбор признаков завершён")
print("Лучшие признаки (индексы):", sfs.k_feature_idx_)

# ====== ОБУЧЕНИЕ НА ОТОБРАННЫХ ПРИЗНАКАХ ======
selected_features = list(sfs.k_feature_idx_)
X_train_sel = X_train_feat[:, selected_features]
X_test_sel = X_test_feat[:, selected_features]

model_final = CatBoostClassifier(
    loss_function="Logloss",
    task_type="CPU",
    random_seed=0,
    iterations=300,
    verbose=False
)

model_final.fit(X_train_sel, y_train)

y_pred_final = model_final.predict(X_test_sel)
y_proba_final = model_final.predict_proba(X_test_sel)[:, 1]

metrics_final = {
    'auc': roc_auc_score(y_test, y_proba_final),
    'f1': f1_score(y_test, y_pred_final),
    'precision': precision_score(y_test, y_pred_final),
    'recall': recall_score(y_test, y_pred_final),
    'logloss': log_loss(y_test, y_proba_final)
}

print("✅ Метрики после отбора признаков:")
print(metrics_final)

# ====== ЛОГИРОВАНИЕ ======
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("churn_project_experiment")

with mlflow.start_run(run_name="feature_selection") as run:
    run_id = run.info.run_id
    mlflow.log_metrics(metrics_final)
    mlflow.log_params({"k_features": 15, "method": "SFS"})

    signature = mlflow.models.infer_signature(X_test_sel, y_pred_final)
    model_info = mlflow.catboost.log_model(
        cb_model=model_final,
        artifact_path="models",
        registered_model_name="churn_model_feature_selection",
        signature=signature,
        input_example=X_test_sel[:5]
    )

    print(f"✅ Модель зарегистрирована")
    print(f"   Run ID: {run_id}")
    print(f"   Версия модели: {model_info.registered_model_version}")

⏳ Запуск SFS... (может занять 5–10 минут)


NameError: name 'X_train_feat' is not defined